In [5]:
import os
import sys
import numpy as np
import moeabench as mb

sys.path.append(os.path.abspath("."))
from src.meamt_dinamico import MEAMT_DIN
from src.mochila import MochilaMultiobjetivo
from src.meamt import MEAMT

MAX_FES = 300000
NOBJ = 3
MIN_TABLES_MEAMT = 25

pop_meamt = MIN_TABLES_MEAMT * (2 ** NOBJ)
gen_meamt = MAX_FES // pop_meamt

prob = mb.mops.DTLZ3(M=NOBJ, N=NOBJ + 10 - 1)

exp_nsga3 = mb.experiment()
exp_nsga3.moea = mb.moeas.NSGA3(population=300, generations=1000)
exp_nsga3.mop = prob

exp_meamt_var = mb.experiment()
exp_meamt_var.moea = MEAMT_DIN(population=pop_meamt, generations= gen_meamt)
exp_meamt_var.mop = prob

exp_meamt = mb.experiment()
exp_meamt.moea = MEAMT(population=pop_meamt, generations= gen_meamt)
exp_meamt.mop = prob


exp_nsga3.run(repeat=1) 
exp_meamt_var.run(repeat=1) 
exp_meamt.run(repeat=1)

### Running **exp_nsga3**

  Run 1/1:   0%|          | 0/1000 [00:00<?, ?gen/s]

### Running **exp_meamt_var**

  Run 1/1:   0%|          | 0/1500 [00:00<?, ?gen/s]

### Running **exp_meamt**

  Run 1/1:   0%|          | 0/1500 [00:00<?, ?gen/s]

In [6]:
Z_ref = prob.pf(1500)
nadir = np.max(Z_ref, axis=0)
limite = nadir * 1.5

F_atual_nsga3 = np.array(exp_nsga3[0].pop().objs)
mascara1 = np.all(F_atual_nsga3 <= limite, axis=1)
F_final_nsga3 = F_atual_nsga3[mascara1]

F_atual_meamtv = np.array(exp_meamt_var[0].pop().objs)
mascara2 = np.all(F_atual_meamtv <= limite, axis=1)
F_final_meamtv = F_atual_meamtv[mascara2]

F_final_meamt = np.array(exp_meamt[0].pop().objs)
mascara_meamt = np.all(F_final_meamt <= limite, axis=1)
F_final_meamt = F_final_meamt[mascara_meamt]


mb.view.topology(F_final_nsga3, F_final_meamtv, F_final_meamt, labels=['NSGA3', 'MEAMT_VAR', 'MEAMT'], show_gt=True, gt=exp_nsga3.optimal_front())


# A Abordagem de Tchebycheff (Tchebycheff Approach)

$$\text{minimizar } g^{te}(x \mid \lambda, z^*) = \max_{1 \leq i \leq M} \{ \lambda_i \cdot \vert{}f_i(x) - z_i^*\vert{} \}$$

Onde:

$x$: É a solução (indivíduo) sendo avaliada.

$\lambda$: É o vetor de pesos direcionais (onde $\lambda_i \geq 0$).

$z^*$: É o Ponto de Referência Ideal, composto pelo melhor valor isolado já encontrado para cada um dos $M$ objetivos ($z_i^* = \min \{f_i(x)\}$).

$M$: É o número total de objetivos do problema